In [ ]:
%pip install -U -q transformers datasets accelerate scikit-learn "protobuf==3.20.3" sentencepiece

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import os
import zipfile
from datasets import load_dataset, Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModel,
    TrainingArguments,
    Trainer,
    TrainerCallback
)
from transformers.models.roberta.modeling_roberta import RobertaPreTrainedModel, RobertaModel
from dataclasses import dataclass
from typing import Optional, Tuple, Any, Dict, List
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = 'cuda' 

In [ ]:
MODEL_NAME = "roberta-large"
MAX_LENGTH = 512
N_FOLDS = 7

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

clarity_label2id = {"Clear Reply": 0, "Ambivalent": 1, "Clear Non-Reply": 2}
clarity_id2label = {v: k for k, v in clarity_label2id.items()}
clarity_labels = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]

evasion_classes = [
    'Declining to answer', 'Dodging', 'General', 'Explicit', 
    'Claims ignorance', 'Clarification', 'Implicit', 
    'Partial/half-answer', 'Deflection'
]
evasion_label2id = {label: idx for idx, label in enumerate(evasion_classes)}
evasion_id2label = {v: k for k, v in evasion_label2id.items()}

In [ ]:
ds = load_dataset("ailsntua/QEvasion")
train_full_df = ds['train'].to_pandas()
dev_df = ds['test'].to_pandas()

print(f"Train columns: {train_full_df.columns.tolist()}")
print(f"Train size: {len(train_full_df)}")
print(f"\nClarity label distribution (train):")
print(train_full_df['clarity_label'].value_counts())
print(f"\nEvasion label distribution (train):")
print(train_full_df['evasion_label'].value_counts())

print(f"\nDev columns: {dev_df.columns.tolist()}")
print(f"Dev size: {len(dev_df)}")
if "clarity_label" in dev_df.columns:
    print(f"\nClarity label distribution (dev):")
    print(dev_df['clarity_label'].value_counts())
else:
    print("\nNo clarity_label column found in dev split.")

annotator_cols = [c for c in ["annotator1", "annotator2", "annotator3"] if c in dev_df.columns]
print(f"\nDev annotator columns for evasion: {annotator_cols if annotator_cols else 'None'}")

clarity_counts = train_full_df['clarity_label'].value_counts()
evasion_counts = train_full_df['evasion_label'].value_counts()

clarity_class_weights = torch.tensor(
    [1.0 / clarity_counts[label] for label in clarity_labels],
    dtype=torch.float
 )
clarity_class_weights = clarity_class_weights / clarity_class_weights.mean()

evasion_class_weights = torch.tensor(
    [1.0 / evasion_counts[label] for label in evasion_classes],
    dtype=torch.float
 )
evasion_class_weights = evasion_class_weights / evasion_class_weights.mean()

print("\nClass weights (inverse frequency, mean-normalized):")
print(f"Clarity: {clarity_class_weights.tolist()}")
print(f"Evasion: {evasion_class_weights.tolist()}")

In [ ]:
def tokenize_function_train(examples):
    inputs = [
        f"Question: {q}\nAnswer: {a}"
        for q, a in zip(examples["question"], examples["interview_answer"])
    ]
    tokenized = tokenizer(
        inputs,
        truncation=False,  
        padding=False,
        max_length=None,
    )
    
    tokenized["clarity_labels"] = [clarity_label2id[l] for l in examples["clarity_label"]]
    tokenized["evasion_labels"] = [evasion_label2id[l] for l in examples["evasion_label"]]
    
    return tokenized


def tokenize_function_dev(examples):
    inputs = [
        f"Question: {q}\nAnswer: {a}"
        for q, a in zip(examples["question"], examples["interview_answer"])
    ]
    tokenized = tokenizer(
        inputs,
        truncation=False, 
        padding=False,
        max_length=None,
    )
    
    if "clarity_label" in examples:
        tokenized["clarity_labels"] = [clarity_label2id[l] for l in examples["clarity_label"]]
    else:
        tokenized["clarity_labels"] = [-100] * len(examples["question"])
    
    # Dev split has no single gold evasion label; mask this head during prediction/eval.
    tokenized["evasion_labels"] = [-100] * len(examples["question"])
    
    return tokenized

train_full_dataset = Dataset.from_pandas(train_full_df, preserve_index=False)
train_tokenized_full = train_full_dataset.map(
    tokenize_function_train, 
    batched=True, 
    remove_columns=train_full_dataset.column_names
)

dev_dataset = Dataset.from_pandas(dev_df, preserve_index=False)
dev_tokenized = dev_dataset.map(
    tokenize_function_dev, 
    batched=True, 
    remove_columns=dev_dataset.column_names
)

print(f"Train tokenized: {len(train_tokenized_full)} samples")
print(f"Dev tokenized: {len(dev_tokenized)} samples")

In [ ]:
@dataclass
class HierarchicalMultiHeadDataCollator:
    tokenizer: Any
    
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        clarity_labels = torch.tensor(
            [f.pop("clarity_labels") for f in features], 
            dtype=torch.long
        )
        evasion_labels = torch.tensor(
            [f.pop("evasion_labels") for f in features], 
            dtype=torch.long
        )
        
        batch = self.tokenizer.pad(
            features,
            padding=True,
            return_tensors="pt"
        )
        batch["clarity_labels"] = clarity_labels
        batch["evasion_labels"] = evasion_labels
        
        return batch

data_collator = HierarchicalMultiHeadDataCollator(tokenizer=tokenizer)

In [ ]:
@dataclass
class MultiHeadClassifierOutput:
    loss: Optional[torch.FloatTensor] = None
    logits_clarity: torch.FloatTensor = None
    logits_evasion: torch.FloatTensor = None
    hidden_states: Optional[Tuple[torch.FloatTensor, ...]] = None
    attentions: Optional[Tuple[torch.FloatTensor, ...]] = None


class MultiHeadHierarchicalMaxPooling(RobertaPreTrainedModel):
    
    @classmethod
    def _can_set_experts_implementation(cls) -> bool:
        return False
    
    def __init__(self, config):
        super().__init__(config)
        self.num_clarity_labels = 3
        self.num_evasion_labels = 9
        self.config = config

        self.roberta = RobertaModel(config)

        classifier_dropout = (
            config.classifier_dropout 
            if hasattr(config, 'classifier_dropout') and config.classifier_dropout is not None 
            else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)

        self.classifier_clarity = nn.Linear(config.hidden_size, self.num_clarity_labels)
        self.classifier_evasion = nn.Linear(config.hidden_size, self.num_evasion_labels)

        self.register_buffer("clarity_class_weights", clarity_class_weights.clone().detach())
        self.register_buffer("evasion_class_weights", evasion_class_weights.clone().detach())

        self.post_init()
    
    def create_chunks_batched(self, input_ids, attention_mask, max_length=512, stride=256):
        batch_size, seq_len = input_ids.shape
        
        all_chunk_ids = []
        all_chunk_masks = []
        chunk_to_sample = [] 
        
        for sample_idx in range(batch_size):
            sample_input_ids = input_ids[sample_idx]
            sample_attention_mask = attention_mask[sample_idx]

            actual_length = sample_attention_mask.sum().item()

            if actual_length <= max_length:
                chunk_ids = sample_input_ids[:max_length]
                chunk_mask = sample_attention_mask[:max_length]
                
                if len(chunk_ids) < max_length:
                    padding_length = max_length - len(chunk_ids)
                    chunk_ids = torch.cat([
                        chunk_ids,
                        torch.zeros(padding_length, dtype=torch.long, device=input_ids.device)
                    ])
                    chunk_mask = torch.cat([
                        chunk_mask,
                        torch.zeros(padding_length, dtype=torch.long, device=attention_mask.device)
                    ])
                
                all_chunk_ids.append(chunk_ids)
                all_chunk_masks.append(chunk_mask)
                chunk_to_sample.append(sample_idx)
            else:
                start = 0
                while start < actual_length:
                    end = min(start + max_length, actual_length)
                    
                    chunk_ids = sample_input_ids[start:end]
                    chunk_mask = sample_attention_mask[start:end]
                    
                    if len(chunk_ids) < max_length:
                        padding_length = max_length - len(chunk_ids)
                        chunk_ids = torch.cat([
                            chunk_ids,
                            torch.zeros(padding_length, dtype=torch.long, device=input_ids.device)
                        ])
                        chunk_mask = torch.cat([
                            chunk_mask,
                            torch.zeros(padding_length, dtype=torch.long, device=attention_mask.device)
                        ])
                    
                    all_chunk_ids.append(chunk_ids)
                    all_chunk_masks.append(chunk_mask)
                    chunk_to_sample.append(sample_idx)
  
                    start += stride
                    if end >= actual_length:
                        break

        all_chunk_ids = torch.stack(all_chunk_ids, dim=0)
        all_chunk_masks = torch.stack(all_chunk_masks, dim=0)
        chunk_to_sample = torch.tensor(chunk_to_sample, dtype=torch.long, device=input_ids.device)
        
        return all_chunk_ids, all_chunk_masks, chunk_to_sample
    
    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        clarity_labels=None,
        evasion_labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ) -> MultiHeadClassifierOutput:
        
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict
        
        batch_size = input_ids.shape[0]

        all_chunk_ids, all_chunk_masks, chunk_to_sample = self.create_chunks_batched(
            input_ids, attention_mask, max_length=512, stride=256
        )

        outputs = self.roberta(
            input_ids=all_chunk_ids,
            attention_mask=all_chunk_masks,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=True,
        )

        all_cls_embeddings = outputs.last_hidden_state[:, 0, :]
        batch_embeddings = []
        for sample_idx in range(batch_size):
            sample_mask = chunk_to_sample == sample_idx
            sample_chunk_embeddings = all_cls_embeddings[sample_mask]  

            pooled_embedding = torch.max(sample_chunk_embeddings, dim=0)[0]  
            batch_embeddings.append(pooled_embedding)

        pooled_output = torch.stack(batch_embeddings, dim=0)
        pooled_output = self.dropout(pooled_output)

        logits_clarity = self.classifier_clarity(pooled_output)
        logits_evasion = self.classifier_evasion(pooled_output)

        loss = None
        if clarity_labels is not None and evasion_labels is not None:
            loss_fct_clarity = nn.CrossEntropyLoss(
                weight=self.clarity_class_weights,
                ignore_index=-100
            )
            loss_fct_evasion = nn.CrossEntropyLoss(
                weight=self.evasion_class_weights,
                ignore_index=-100
            )
            loss_clarity = loss_fct_clarity(logits_clarity.view(-1, self.num_clarity_labels), clarity_labels.view(-1))
            loss_evasion = loss_fct_evasion(logits_evasion.view(-1, self.num_evasion_labels), evasion_labels.view(-1))
            loss = loss_clarity + loss_evasion
        
        return MultiHeadClassifierOutput(
            loss=loss,
            logits_clarity=logits_clarity,
            logits_evasion=logits_evasion,
            hidden_states=None,
            attentions=None,
        )

In [ ]:
def compute_metrics(eval_pred):
    predictions = eval_pred.predictions
    labels = eval_pred.label_ids

    if isinstance(predictions, tuple):
        logits_clarity, logits_evasion = predictions
        if hasattr(logits_clarity, 'cpu'):
            logits_clarity = logits_clarity.cpu().numpy()
        if hasattr(logits_evasion, 'cpu'):
            logits_evasion = logits_evasion.cpu().numpy()
    else:
        logits_clarity = predictions[:, :3]
        logits_evasion = predictions[:, 3:]
    preds_clarity = np.argmax(logits_clarity, axis=-1)
    preds_evasion = np.argmax(logits_evasion, axis=-1)
    if isinstance(labels, tuple):
        labels_clarity, labels_evasion = labels
        if hasattr(labels_clarity, 'cpu'):
            labels_clarity = labels_clarity.cpu().numpy()
        if hasattr(labels_evasion, 'cpu'):
            labels_evasion = labels_evasion.cpu().numpy()
    else:
        labels_clarity = labels[:, 0] if labels.ndim > 1 else labels
        labels_evasion = labels[:, 1] if labels.ndim > 1 else labels

    valid_evasion_mask = labels_evasion != -100

    accuracy_clarity = accuracy_score(labels_clarity, preds_clarity)
    f1_clarity = f1_score(labels_clarity, preds_clarity, average='macro')

    if valid_evasion_mask.sum() > 0:
        accuracy_evasion = accuracy_score(labels_evasion[valid_evasion_mask], preds_evasion[valid_evasion_mask])
        f1_evasion = f1_score(labels_evasion[valid_evasion_mask], preds_evasion[valid_evasion_mask], average='macro')
    else:
        accuracy_evasion = 0.0
        f1_evasion = 0.0
    
    return {
        'accuracy_clarity': accuracy_clarity,
        'f1_clarity': f1_clarity,
        'accuracy_evasion': accuracy_evasion,
        'f1_evasion': f1_evasion,
        'f1_combined': (f1_clarity + f1_evasion) / 2,
    }

In [ ]:
class MultiHeadTrainer(Trainer):
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        loss = outputs.loss
        return (loss, outputs) if return_outputs else loss
    
    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)
        clarity_labels = inputs.get('clarity_labels')
        evasion_labels = inputs.get('evasion_labels')
        
        with torch.no_grad():
            outputs = model(**inputs)
            loss = outputs.loss
            logits_clarity = outputs.logits_clarity
            logits_evasion = outputs.logits_evasion
        
        if prediction_loss_only:
            return (loss, None, None)
        
        logits = (logits_clarity.detach().float(), logits_evasion.detach().float())

        if clarity_labels is not None and evasion_labels is not None:
            labels = (clarity_labels.detach(), evasion_labels.detach())
        else:
            labels = None
        
        return (loss, logits, labels)


class EarlyStoppingCallback(TrainerCallback):    
    def __init__(self, patience: int = 3, metric_name: str = "eval_f1_combined", greater_is_better: bool = True):
        self.patience = patience
        self.metric_name = metric_name
        self.greater_is_better = greater_is_better
        self.best_metric = None
        self.patience_counter = 0
        
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        current_metric = metrics.get(self.metric_name)
        
        if current_metric is None:
            return
        
        if self.best_metric is None:
            self.best_metric = current_metric
            self.patience_counter = 0
        else:
            if self.greater_is_better:
                improved = current_metric > self.best_metric
            else:
                improved = current_metric < self.best_metric
            
            if improved:
                self.best_metric = current_metric
                self.patience_counter = 0
            else:
                self.patience_counter += 1
                
        if self.patience_counter >= self.patience:
            print(f"\nEarly stopping triggered!")
            print(f"Best {self.metric_name}: {self.best_metric:.4f}")
            control.should_training_stop = True

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

fold_dev_probs_clarity = []
fold_dev_probs_evasion = []

fold_oof_probs_clarity = []
fold_oof_probs_evasion = []
fold_oof_indices = []

fold_metrics = []

best_fold = None
best_f1_combined = 0
best_model_state = None

for fold, (train_idx, val_idx) in enumerate(skf.split(train_full_df, train_full_df['clarity_label']), 1):
    print(f"\nFold {fold}/{N_FOLDS}")
    print(f"{'='*30}")
    
    train_dataset = train_tokenized_full.select(train_idx)
    val_dataset = train_tokenized_full.select(val_idx)
    
    print(f"Train samples: {len(train_dataset)}")
    print(f"Val samples: {len(val_dataset)}")
    
    config = AutoConfig.from_pretrained(MODEL_NAME)
    
    model = MultiHeadHierarchicalMaxPooling(config)
    
    base_model = AutoModel.from_pretrained(MODEL_NAME)
    model.roberta.load_state_dict(base_model.state_dict())
    del base_model
    
    training_args = TrainingArguments(
        output_dir=f"./results_multihead_maxpool_class_weighted_fold{fold}",
        learning_rate=1e-5,
        per_device_train_batch_size=8, 
        per_device_eval_batch_size=8,
        num_train_epochs=15,
        warmup_ratio=0.1,
        weight_decay=0.01,
        max_grad_norm=1.0,
        gradient_checkpointing=True,
        bf16=True,
        bf16_full_eval=True,
        optim="adamw_torch",
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1_combined",
        greater_is_better=True,
        logging_steps=50,
        seed=SEED + fold,
        disable_tqdm=True,
        report_to="none",
    )
    
    trainer = MultiHeadTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(patience=3, metric_name="eval_f1_combined")],
    )

    trainer.train()

    val_results = trainer.evaluate()
    fold_metrics.append({
        'fold': fold,
        'val_accuracy_clarity': val_results['eval_accuracy_clarity'],
        'val_f1_clarity': val_results['eval_f1_clarity'],
        'val_accuracy_evasion': val_results['eval_accuracy_evasion'],
        'val_f1_evasion': val_results['eval_f1_evasion'],
        'val_f1_combined': val_results['eval_f1_combined'],
    })
    
    if val_results['eval_f1_combined'] > best_f1_combined:
        best_f1_combined = val_results['eval_f1_combined']
        best_fold = fold
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    
    print(f"Fold {fold} Results:")
    print(f"  Clarity - Accuracy: {val_results['eval_accuracy_clarity']:.4f}, F1: {val_results['eval_f1_clarity']:.4f}")
    print(f"  Evasion - Accuracy: {val_results['eval_accuracy_evasion']:.4f}, F1: {val_results['eval_f1_evasion']:.4f}")
    print(f"  Combined F1: {val_results['eval_f1_combined']:.4f}")
    
    oof_output = trainer.predict(val_dataset)
    logits_clarity_oof, logits_evasion_oof = oof_output.predictions

    oof_probs_clarity = torch.nn.functional.softmax(torch.tensor(logits_clarity_oof), dim=-1).numpy()
    oof_probs_evasion = torch.nn.functional.softmax(torch.tensor(logits_evasion_oof), dim=-1).numpy()
    
    fold_oof_probs_clarity.append(oof_probs_clarity)
    fold_oof_probs_evasion.append(oof_probs_evasion)
    fold_oof_indices.append(val_idx)

    dev_output = trainer.predict(dev_tokenized)
    logits_clarity_dev, logits_evasion_dev = dev_output.predictions
    
    dev_probs_clarity = torch.nn.functional.softmax(torch.tensor(logits_clarity_dev), dim=-1).numpy()
    dev_probs_evasion = torch.nn.functional.softmax(torch.tensor(logits_evasion_dev), dim=-1).numpy()
    
    fold_dev_probs_clarity.append(dev_probs_clarity)
    fold_dev_probs_evasion.append(dev_probs_evasion)
    
    del model, trainer
    torch.cuda.empty_cache()

print(f"\nBest fold: {best_fold} with F1 Combined: {best_f1_combined:.4f}")

In [ ]:
print("Cross-Validation Metrics")

metrics_df = pd.DataFrame(fold_metrics)
print(metrics_df.to_string(index=False))

print(f"\nClarity Head:")
print(f"Average Validation Accuracy: {metrics_df['val_accuracy_clarity'].mean():.4f} ± {metrics_df['val_accuracy_clarity'].std():.4f}")
print(f"Average Validation Macro F1: {metrics_df['val_f1_clarity'].mean():.4f} ± {metrics_df['val_f1_clarity'].std():.4f}")

print(f"\nEvasion Head:")
print(f"Average Validation Accuracy: {metrics_df['val_accuracy_evasion'].mean():.4f} ± {metrics_df['val_accuracy_evasion'].std():.4f}")
print(f"Average Validation Macro F1: {metrics_df['val_f1_evasion'].mean():.4f} ± {metrics_df['val_f1_evasion'].std():.4f}")

print(f"\nCombined:")
print(f"Average Combined F1: {metrics_df['val_f1_combined'].mean():.4f} ± {metrics_df['val_f1_combined'].std():.4f}")

In [ ]:
print("Full training set OOF analysis")

oof_probs_clarity_full = np.zeros((len(train_full_df), 3))
oof_probs_evasion_full = np.zeros((len(train_full_df), 9))

for probs_c, probs_e, idxs in zip(fold_oof_probs_clarity, fold_oof_probs_evasion, fold_oof_indices):
    oof_probs_clarity_full[idxs] = probs_c
    oof_probs_evasion_full[idxs] = probs_e

oof_preds_clarity = np.argmax(oof_probs_clarity_full, axis=1)
oof_preds_evasion = np.argmax(oof_probs_evasion_full, axis=1)

oof_pred_labels_clarity = [clarity_id2label[p] for p in oof_preds_clarity]
oof_pred_labels_evasion = [evasion_id2label[p] for p in oof_preds_evasion]

y_true_train_clarity = train_full_df['clarity_label'].tolist()
y_true_train_evasion = train_full_df['evasion_label'].tolist()

print("Clarity head (OOF)")
print(classification_report(y_true_train_clarity, oof_pred_labels_clarity, labels=clarity_labels, zero_division=0))

print("\nEvasion head (OOF)")
print(classification_report(y_true_train_evasion, oof_pred_labels_evasion, labels=evasion_classes, zero_division=0))

In [ ]:
print("Ensemble predictions for QEvasion dev split")
print("="*50)

ensemble_probs_clarity = np.mean(fold_dev_probs_clarity, axis=0)
ensemble_probs_evasion = np.mean(fold_dev_probs_evasion, axis=0)

ensemble_preds_clarity = np.argmax(ensemble_probs_clarity, axis=1)
ensemble_preds_evasion = np.argmax(ensemble_probs_evasion, axis=1)

ensemble_labels_clarity = [clarity_id2label[p] for p in ensemble_preds_clarity]
ensemble_labels_evasion = [evasion_id2label[p] for p in ensemble_preds_evasion]

print(f"Total predictions: {len(ensemble_labels_clarity)}")
print(f"\nClarity predictions distribution:")
print(pd.Series(ensemble_labels_clarity).value_counts())
print(f"\nEvasion predictions distribution:")
print(pd.Series(ensemble_labels_evasion).value_counts())

if "clarity_label" in dev_df.columns:
    valid_clarity_mask = dev_df["clarity_label"].isin(clarity_label2id.keys())
    if valid_clarity_mask.any():
        y_true_dev_clarity = dev_df.loc[valid_clarity_mask, "clarity_label"].tolist()
        y_pred_dev_clarity = np.array(ensemble_labels_clarity)[valid_clarity_mask.values].tolist()
        print("\nDev clarity evaluation")
        print(f"Accuracy: {accuracy_score(y_true_dev_clarity, y_pred_dev_clarity):.4f}")
        print(f"Macro F1: {f1_score(y_true_dev_clarity, y_pred_dev_clarity, average='macro', labels=clarity_labels, zero_division=0):.4f}")
        print("\nClassification Report:")
        print(classification_report(y_true_dev_clarity, y_pred_dev_clarity, labels=clarity_labels, zero_division=0))

annotator_cols = [c for c in ["annotator1", "annotator2", "annotator3"] if c in dev_df.columns]
if annotator_cols:
    pred_evasion_arr = np.array(ensemble_labels_evasion)
    agreement_any = []
    for idx, pred in enumerate(pred_evasion_arr):
        row = dev_df.iloc[idx]
        ann_set = {
            row[c] for c in annotator_cols
            if isinstance(row[c], str) and row[c] in evasion_label2id
        }
        agreement_any.append(pred in ann_set if ann_set else False)
    print(f"\nEvasion any-annotator agreement (dev): {np.mean(agreement_any):.4f}")

    for col in annotator_cols:
        valid_mask = dev_df[col].apply(lambda x: isinstance(x, str) and x in evasion_label2id)
        if valid_mask.any():
            ann_acc = accuracy_score(dev_df.loc[valid_mask, col], pred_evasion_arr[valid_mask.values])
            print(f"Evasion agreement vs {col}: {ann_acc:.4f}")
else:
    print("\nNo annotator columns found in dev split for evasion agreement analysis.")

In [ ]:
os.makedirs("../../results", exist_ok=True)

dev_predictions_df = dev_df.copy()
dev_predictions_df["pred_clarity"] = ensemble_labels_clarity
dev_predictions_df["pred_evasion"] = ensemble_labels_evasion

dev_predictions_path = "../../results/qevasion_dev_predictions_multihead_maxpool_kfold_class_weighted.csv"
dev_predictions_df.to_csv(dev_predictions_path, index=False)

print(f"Saved dev predictions to: {dev_predictions_path}")
print(f"Columns saved: {dev_predictions_df.columns.tolist()}")
print(f"Rows saved: {len(dev_predictions_df)}")

In [ ]:
print(f"Saving best model (from Fold {best_fold})...")

model_save_path = "../../models/multihead_maxpool_kfold_best_class_weighted"
os.makedirs(model_save_path, exist_ok=True)

config = AutoConfig.from_pretrained(MODEL_NAME)
best_model = MultiHeadHierarchicalMaxPooling(config)
best_model.load_state_dict(best_model_state)

torch.save(best_model.state_dict(), os.path.join(model_save_path, "pytorch_model.bin"))

config.save_pretrained(model_save_path)

tokenizer.save_pretrained(model_save_path)

import json
label_mappings = {
    "clarity_label2id": clarity_label2id,
    "clarity_id2label": clarity_id2label,
    "evasion_label2id": evasion_label2id,
    "evasion_id2label": evasion_id2label,
}
with open(os.path.join(model_save_path, "label_mappings.json"), 'w') as f:
    json.dump(label_mappings, f, indent=2)

training_info = {
    "best_fold": best_fold,
    "best_f1_combined": best_f1_combined,
    "n_folds": N_FOLDS,
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "num_clarity_labels": 3,
    "num_evasion_labels": 9,
    "loss_type": "class_weighted_cross_entropy",
    "clarity_class_weights": clarity_class_weights.tolist(),
    "evasion_class_weights": evasion_class_weights.tolist(),
    "inference_split": "QEvasion test split (used as dev)",
}
with open(os.path.join(model_save_path, "training_info.json"), 'w') as f:
    json.dump(training_info, f, indent=2)

print(f"Model saved to: {model_save_path}")
print(f"Files saved:")
for f in os.listdir(model_save_path):
    print(f"  - {f}")